In [1]:
import requests
import os
import time
import zipfile

## Funciones para descargar datos

In [2]:
def download_rama_2(year, month, parameter, download_folder="downloads"):
    base_url = "http://www.aire.cdmx.gob.mx/estadisticas-consultas/concentraciones/respuesta.php"
    download_url = "http://www.aire.cdmx.gob.mx/estadisticas-consultas/concentraciones/p_descarga.php?um=Promedio%20horarios%20de%20{}".format(parameter)
    
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)
    
    params = {
        "qtipo": "HORARIOS",
        "parametro": parameter,
        "anio": year,
        "qmes": month
    }
    
    # Try to download the file 5 times before failing
    for i in range(5):
        try:
            
            # Open session to maintain the cookie
            session = requests.Session()
            session.get(base_url, params=params)
            
            response = session.get(download_url, stream=True)
            if response.status_code == 200:
                file_name = os.path.join(download_folder, f"{year}_{month:02d}_{parameter}.csv")
                with open(file_name, "wb") as file:
                    for chunk in response.iter_content(chunk_size=1024):
                        file.write(chunk)
                print(f"File downloaded: {file_name}")
                time.sleep(0.5)
                break
            
        except Exception as e:
            print(f"Error downloading {year}-{month:02d} {parameter}")
            # wait 5 seconds before retrying
            time.sleep(5)


In [3]:
def download_file(url, download_folder, file_name, unzip=False):
    
    for attempt in range(5):
        
        try:
            response = requests.get(url, stream=True)
            if response.status_code == 200:
                if not os.path.exists(download_folder):
                    os.makedirs(download_folder)
                with open(f'{download_folder}/{file_name}', "wb") as file:
                    for chunk in response.iter_content(chunk_size=1024):
                        file.write(chunk)
                print(f"Archivo descargado: {file_name}")
                if unzip:
                    print("Descomprimiendo archivo")
                    with zipfile.ZipFile(os.path.join(download_folder, file_name), 'r') as zip_ref:
                        zip_ref.extractall(download_folder)
                break
            else:
                print(f"Intento {attempt+1}, Error al descargar: {url} (status code: {response.status_code})")
        except Exception as e:
            print(f"Error descargando {url}: {e}")
        
        # wait 5 seconds before retrying
        time.sleep(5)

# Datos RAMA Mensual 2005-hoy

In [6]:
pollutants = ['so2','co','nox','no2','no','o3','pm10','pm2','wsp','wdr','tmp','rh']
start_year = 2026
current_year = time.localtime().tm_year
current_month = time.localtime().tm_mon

In [7]:
download_path = "../data/cdmx/raw/RAMA2/"

for year in range(start_year, current_year + 1):  # Iterate over the years
    for month in range(1, 13):  # Iterate over the 12 months
        for pollutant in pollutants:
            if year == current_year and month >= current_month:
                break
            download_rama_2(year, month, pollutant, download_path)


File downloaded: ../data/cdmx/raw/RAMA2/2026_01_so2.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_co.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_nox.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_no2.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_no.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_o3.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_pm10.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_pm2.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_wsp.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_wdr.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_tmp.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_01_rh.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_02_so2.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_02_co.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_02_nox.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_02_no2.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_02_no.csv
File downloaded: ../data/cdmx/raw/RAMA2/2026_02_o3.cs

# Datos RAMA/REDMET/REDMA/REDDA

Estos archivos se descargan desde la página: http://datosabiertos.aire.cdmx.gob.mx:8080/opendata/Bases_publicas.

Cada una de las bases se descarga por año en formato `.zip `, y el año de inicio es diferente para cada uno.

In [ ]:
bases = {
    'RAMA': {'inicio': 1986, 'fin': 2025},
    'REDMET': {'inicio': 1986, 'fin': 2025},
    'REDMA': {'inicio': 1989, 'fin': 2025},
    'REDDA': {'inicio': 1988, 'fin': 2023},
}

In [ ]:
base_url = "http://datosabiertos.aire.cdmx.gob.mx:8080/opendata/Bases_publicas"
download_folder = "../data/cdmx/raw"


for param, years in bases.items():
    for year in range(years['inicio'], years['fin'] + 1):
        url = f"{base_url}/{param}/{year % 100:02d}{param}.zip"
        file_name = str(year) + '_' + param + '.zip'
        download_file(url, download_folder + f'/{param}', file_name, unzip=True)
